# Dual-LLM label re-extraction (Claude + ChatGPT/OpenAI, or Gemini) with a gold-convention rubric
CPU only (no GPU needed). Runs on Colab or anywhere with Python.

**What it does**
1. Loads the 4,407 training reports (`train_processed_v3_1.csv`) and the hand labels (silver-1 60, silver-2 159) from the Kaggle dataset `harrisonhoran/rsna-knee-translatedprocessed-train-files`.
2. Sends each report to **Claude** and **OpenAI** (or Gemini) with the same rubric: the official label criteria plus calibration anchors (report wording → probability that the image readers marked the finding positive). Output per label: probability `p`, status, evidence quote; plus a graded block (effusion size, ACL/MCL state, meniscus state, cartilage grade per compartment...) for auxiliary targets later.
3. **PILOT** (`STAGE = "pilot"`): 58 gold + 159 silver-2 reports only. Scores each model, and their average, against gold (blind: the model never sees the gold label) and against the silver-2 hand labels, next to the current v3.1 extraction. Cheap (~$2–5).
4. **FULL** (`STAGE = "full"`): all 4,407 reports, then writes `llm_<model>.csv` per model, `llm_combined.csv` and `llm_disagreements.csv` to Drive. Send those to Claude (the chat) to build the training labels.

**Keys.** Colab: Secrets (key icon) `ANTHROPIC_API_KEY`, `GEMINI_API_KEY` (Google AI Studio), `KAGGLE_API_TOKEN`. **Local (VS Code):** put `ANTHROPIC_API_KEY=...` and `OPENAI_API_KEY=...` (or `GEMINI_API_KEY=...`) in a `.env` file next to this notebook (never commit it); inputs come from `LOCAL_DATASET_DIR`, so no Kaggle key is needed.

**Resumable:** every answer is appended to a `.jsonl` in Drive as it arrives; re-running skips finished reports. Safe to stop and restart.

**Rule for an honest check:** run the pilot **once** with the rubric as written. Changing the rubric until the gold score looks good would leak the gold labels into the prompt.

In [ ]:
# ---- setup -------------------------------------------------------------------
import subprocess, sys, os, json, time, threading, random, re
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic", "openai", "google-genai", "kagglehub"], check=True)
from pathlib import Path
import numpy as np, pandas as pd

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import userdata, drive
    for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/rsna_llm_labels")
else:
    # local (VS Code etc.): API keys from the environment or from a .env file next to this notebook
    for envf in (Path(".env"), Path("..") / ".env"):
        if envf.exists():
            for line in envf.read_text().splitlines():
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
    OUT = Path("rsna_llm_labels")
OUT.mkdir(parents=True, exist_ok=True)
print("keys present:", [k for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "KAGGLE_API_TOKEN", "KAGGLE_KEY") if os.environ.get(k)])
print("outputs ->", OUT)

In [ ]:
# ---- settings ------------------------------------------------------------------
STAGE = "full"                         # "pilot" (gold + silver-2, ~217 reports) or "full" (all 4,407)
CLAUDE_MODEL = "claude-sonnet-5-5"      # or "claude-opus-5-5" (better, ~2x cost) / "claude-haiku-4-5" (cheapest)
OPENAI_MODEL = "gpt-6.1-sol"            # exact model id from platform.openai.com; "gpt-6-luna" = cheap, "gpt-6-astra" = top tier
GEMINI_MODEL = "gemini-2.5-pro"         # only if you use Gemini
MODELS = ["claude", "openai"]           # any two of "claude", "openai", "gemini" (or just one)
WORKERS = 12                             # parallel requests per model; lower it if you hit rate limits
MAX_CHARS = 7000                        # per report text (original + English translation each)
DATASET = "harrisonhoran/rsna-knee-translatedprocessed-train-files"
# local run: a folder holding train_processed_v3_1.csv, silver1_final_labels.csv, silver2_hand_labels.csv
# (leave "" to download the Kaggle dataset instead)
LOCAL_DATASET_DIR = "/Users/harrisonhoran/Documents/projects/kaggle/competitions/comp-rsna-knees/data/llm_inputs"

In [ ]:
# ---- data ------------------------------------------------------------------------
import kagglehub
LOCAL = LOCAL_DATASET_DIR or os.environ.get("LOCAL_DATASET_DIR", "")
DS = Path(LOCAL) if LOCAL and Path(LOCAL).exists() else Path(kagglehub.dataset_download(DATASET))
print("reading inputs from", DS)

def find(name):
    hits = sorted(DS.rglob(name))
    assert hits, f"{name} not found under {DS}"
    return hits[0]

COMP = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
        "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
v31 = pd.read_csv(find("train_processed_v3_1.csv")).set_index("StudyInstanceUID")
s1 = pd.read_csv(find("silver1_final_labels.csv")).set_index("StudyInstanceUID")[COMP]
s2 = pd.read_csv(find("silver2_hand_labels.csv")).set_index("StudyInstanceUID")[COMP]
GOLD = v31.index[v31["label_type"] == "gold"]
print(f"{len(v31):,} reports | gold {len(GOLD)} | silver-1 {len(s1)} | silver-2 {len(s2)} | label types {v31['label_type'].value_counts().to_dict()}")

def report_text(uid):
    r = v31.loc[uid]
    lang = str(r.get("ReportLanguage", "")).strip()
    orig = str(r.get("Report", "") or "")[:MAX_CHARS]
    eng = str(r.get("report_english", "") or "")[:MAX_CHARS]
    parts = [f"REPORT (language: {lang or 'unknown'}):\n{orig}"]
    if eng and eng != "nan" and lang.lower() not in ("en", "english") and eng.strip() != orig.strip():
        parts.append("ENGLISH MACHINE TRANSLATION (prefer the original wording if they conflict):\n" + eng)
    return "\n\n".join(parts)

PILOT_IDS = list(GOLD) + [u for u in s2.index if u in v31.index]
ALL_IDS = list(v31.index)
IDS = PILOT_IDS if STAGE == "pilot" else ALL_IDS
print(f"stage {STAGE}: {len(IDS):,} reports to extract")
print(report_text(IDS[0])[:600])

In [ ]:
# ---- rubric + output schema (identical for both models) -----------------------------
LABELS = ["acl", "mcl", "medial_meniscus", "lateral_meniscus", "medial_oa", "lateral_oa", "pf_oa",
          "effusion", "synovitis", "bakers", "contusion", "fracture"]
COMP = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
        "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

SYSTEM = r"""You convert one knee MRI radiology report into 12 probabilities for a research dataset.

## The question you answer for each finding
Two subspecialty musculoskeletal radiologists later looked at THE IMAGES of this exam and labelled 12 findings using the criteria below (a third radiologist adjudicated). You only see the report. For each finding, estimate the PROBABILITY (0.00-1.00) that those radiologists marked it POSITIVE.
This is a calibrated probability, not a yes/no. Report wording is an imperfect signal of what the images show: reporting radiologists use words like "mild", "small" or "grade 2" loosely, and the image readers sometimes call such findings positive. Use the calibration anchors below; they were measured by comparing report wording with the image readers' labels.

## Official criteria the image readers used (verbatim intent)
General: one label per finding for the whole exam, single knee. Borderline / "on the fence" findings were graded NEGATIVE.
- ACL: positive = high-grade partial or full-thickness tear (complete discontinuity, or >50% of fibers disrupted), with or without pivot-shift contusions. Negative = mild signal change, degeneration or thickening without discontinuity (e.g. mucoid degeneration).
- MCL: positive = high-grade partial or complete ACUTE tear: thickening with edema WITHIN and ADJACENT to the ligament, or torn fibers. Negative = low-grade sprain; chronic/remote changes (thickening or signal without adjacent edema, Pellegrini-Stieda, "old/remote tear").
- Medial meniscus / Lateral meniscus (scored separately): positive = abnormal signal definitely reaching the meniscal surface on at least 2 images, OR a morphologic abnormality (truncated, diminutive, macerated, displaced fragment, bucket handle, flap, root tear). Negative = intrasubstance degeneration not reaching the surface; fraying on one image.
- Medial OA / Lateral OA / PF OA (patellofemoral = patella + trochlea), scored per compartment: positive = high-grade cartilage loss (>50% of thickness) over a moderate/large area (~1 cm or more), with or without subchondral marrow change. Negative = smaller or lower-grade thinning, normal signal heterogeneity.
- Effusion: positive = moderate or large joint fluid distending the joint. Negative = small / physiologic.
- Synovitis: positive = inflammation/thickening of the synovial lining (distinct from fluid volume).
- Baker's cyst: positive = fluid collection in the semimembranosus-medial gastrocnemius interval (popliteal cyst). Mimics are NOT Baker's: parameniscal/meniscal cyst, ganglion, popliteus sheath fluid.
- Contusion: positive = bone marrow edema from impact WITHOUT a discrete fracture line.
- Fracture: positive = acute cortical break or fracture line (incl. avulsion, impaction, osteochondral fracture with bone).

## Calibration anchors (probability the image readers marked POSITIVE, given the report wording)
Use these as defaults and adjust within about +/-0.1 for strong supporting or conflicting details elsewhere in the report.
- Not mentioned anywhere, in a report that describes the relevant structure as normal or clearly covers it: 0.03.
- Not mentioned, in a SHORT impression-only report that skips the structure: ACL/fracture/contusion/Baker's 0.05; MCL, menisci, OA compartments 0.12 (findings are sometimes left out of short reports).
- ACL: complete tear / rupture / high-grade partial / ">50%" / non-visualised ACL / torn graft 0.95. "Partial tear" without grade 0.45. Low-grade / sprain / interstitial 0.20. Mucoid or eosinophilic degeneration, thickened with signal, ganglion 0.25. Intact reconstruction graft 0.05.
- MCL: complete or high-grade (grade 3 / grade 2-3) tear with edema 0.85. Partial tear (grade 2) with edema 0.65. Sprain / grade 1 / periligamentous edema only 0.15. Chronic thickening, old/remote injury, Pellegrini-Stieda 0.07. Proximal "MCL edema" only (edema without ligament abnormality) 0.12.
- Meniscus (each side separately; check medial vs lateral carefully, including "internal" = medial and "external" = lateral in some languages): definite tear (any type), tear reaching the surface, maceration, loss of substance, displaced fragment, bucket handle, root tear, parameniscal cyst with tear 0.92. "Probable/likely/suspected tear", "signal possibly reaching the surface" 0.55. Grade 2 / intrasubstance or degenerative signal NOT reaching surface 0.35. Postoperative meniscus without new tear 0.30; with "re-tear" 0.85. Normal 0.03.
- OA (each compartment separately; only the compartment named): full-thickness / grade 4 / high-grade / severe / bone exposed / denuded / "advanced arthrosis" 0.80. Grade 3, deep partial-thickness (>50%), moderate-to-severe 0.75. Moderate / grade 2 / partial thickness without depth, osteoarthritis "moderate" 0.55. Mild / superficial / grade 1-2 / "chondromalacia" or "chondropathy" without grade / fibrillation / fissuring: PF compartment 0.45, medial or lateral compartment 0.25. Osteophytes only with cartilage described as preserved 0.15. Small focal full-thickness defect (<1 cm, e.g. "focal defect 5 mm") 0.40. Cartilage normal 0.03.
- Effusion: moderate / large / massive / significant / hemarthrosis / lipohemarthrosis 0.85. Small / mild / "some" / unsized "effusion" 0.45. Trace / minimal / physiologic 0.25. None 0.04.
- Synovitis: synovitis, synovial thickening or proliferation named (incl. "reactive synovitis", PVNS, Hoffa synovitis) 0.75. Not mentioned: with moderate/large effusion 0.55; with small or unsized effusion 0.40; with Hoffa fat-pad edema/inflammation and no effusion 0.30; with no effusion 0.08. Trauma knees (acute ACL/MCL tear, fracture, contusion) with an effusion but no synovitis mentioned: subtract 0.15.
- Baker's: moderate / large / "Baker's cyst" without size / popliteal cyst / "fluid in the gastrocnemius-semimembranosus bursa" 0.85. Small 0.85. Trace / tiny / millimetric / "minimal fluid in the bursa" 0.50. Ruptured Baker's cyst 0.85. Only a mimic named 0.08. None 0.02.
- Contusion: bone bruise / contusion / post-traumatic marrow edema, no fracture described 0.80. Same, but a fracture is also described 0.35. Marrow edema that is degenerative, subchondral (next to cartilage loss), stress reaction, or "reactive" 0.15. None 0.03.
- Fracture: acute fracture / fracture line / avulsion fracture (incl. Segond, tibial spine) / osteochondral fracture 0.88. Impaction fracture / "impacted" 0.60. "Possible/subtle/incomplete fracture", "trabecular microfracture" 0.45. Old/healed fracture, stress reaction without a line, bipartite patella 0.08. None 0.03.

## Rules
- Use ONLY this report. Do not infer findings that are not supported by its text, except the stated "not mentioned" anchors.
- Negations: read each finding's own clause ("no tear of the medial meniscus; the lateral meniscus shows a radial tear" => medial ~0.03, lateral ~0.92).
- Laterality inside the knee (medial vs lateral) must come from the report. Never swap compartments.
- If the report covers two knees or is not a knee MRI, set every probability to 0.5 and status "unclear".
- status per finding: "positive" (stated present), "negative" (stated absent/normal), "hedged" (possible/probable/mild/small/borderline wording), "not_mentioned".
- evidence: the shortest quote (max 20 words, original language allowed) that drove the probability, or "" if not mentioned.
- Also fill the "grades" block (use "not_mentioned" when the report is silent).
Answer only through the required JSON structure."""

GRADE_ENUMS = {
    "effusion_size": ["none", "trace_physiologic", "small_mild", "moderate", "large", "not_mentioned"],
    "bakers_size": ["none", "trace_tiny", "small", "moderate_large", "unsized", "not_mentioned"],
    "acl_state": ["intact", "low_grade_sprain", "mucoid_degeneration", "partial_unspecified", "high_grade_partial",
                  "complete", "reconstructed_intact", "reconstructed_torn", "not_mentioned"],
    "mcl_state": ["intact", "sprain_low_grade", "partial_tear", "high_grade_or_complete", "chronic_remote", "not_mentioned"],
    "medial_meniscus_state": ["normal", "degeneration_no_surface", "probable_tear", "tear", "postoperative", "not_mentioned"],
    "lateral_meniscus_state": ["normal", "degeneration_no_surface", "probable_tear", "tear", "postoperative", "not_mentioned"],
    "cartilage_medial": ["normal", "mild_low_grade", "moderate", "high_grade_full_thickness", "not_mentioned"],
    "cartilage_lateral": ["normal", "mild_low_grade", "moderate", "high_grade_full_thickness", "not_mentioned"],
    "cartilage_pf": ["normal", "mild_low_grade", "moderate", "high_grade_full_thickness", "not_mentioned"],
    "synovitis_state": ["none", "mentioned_mild", "mentioned_moderate_severe", "not_mentioned"],
    "marrow_edema_cause": ["none", "traumatic_contusion", "fracture", "degenerative_subchondral", "other", "not_mentioned"],
    "mechanism": ["pivot_shift", "dashboard", "hyperextension", "valgus", "patellar_dislocation", "none_or_unclear"],
}
STATUS = ["positive", "negative", "hedged", "not_mentioned", "unclear"]

def json_schema():
    lab = {"type": "object",
           "properties": {"status": {"type": "string", "enum": STATUS},
                          "p": {"type": "number"},
                          "evidence": {"type": "string"}},
           "required": ["status", "p", "evidence"]}
    return {"type": "object",
            "properties": {"report_is_single_knee_mri": {"type": "boolean"},
                           "labels": {"type": "object", "properties": {k: lab for k in LABELS}, "required": LABELS},
                           "grades": {"type": "object",
                                      "properties": {k: {"type": "string", "enum": v} for k, v in GRADE_ENUMS.items()},
                                      "required": list(GRADE_ENUMS)}},
            "required": ["report_is_single_knee_mri", "labels", "grades"]}


SCHEMA = json_schema()
print(f"system prompt {len(SYSTEM):,} chars; schema fields: {len(LABELS)} labels + {len(GRADE_ENUMS)} grades")

In [ ]:
# ---- model callers ------------------------------------------------------------------
def validate(d):
    # model JSON -> flat dict; raises on anything missing so the call is retried
    if isinstance(d, str):
        d = json.loads(d)
    labs, grades = d["labels"], d.get("grades", {}) or {}
    row = {"single_knee": bool(d.get("report_is_single_knee_mri", True))}
    for k in LABELS:
        x = labs[k]
        p = float(x["p"])
        if not np.isfinite(p):
            raise ValueError(f"non-finite p for {k}")
        row[f"p_{k}"] = min(1.0, max(0.0, p))
        st = str(x.get("status", "")).strip()
        row[f"st_{k}"] = st if st in STATUS else "unclear"
        row[f"ev_{k}"] = str(x.get("evidence", ""))[:300]
    for k, allowed in GRADE_ENUMS.items():
        g = grades.get(k)
        row[f"g_{k}"] = g if g in allowed else None
    return row

_claude = _gemini = None
_claude_variant = 0
def call_claude(text):
    global _claude
    import anthropic
    _claude = _claude or anthropic.Anthropic()
    kw = dict(model=CLAUDE_MODEL, max_tokens=4000,
              system=[{"type": "text", "text": SYSTEM + "\n\nAlways answer by calling the record_labels tool exactly once.",
                       "cache_control": {"type": "ephemeral"}}],
              tools=[{"name": "record_labels", "description": "Record the 12 probabilities, statuses, evidence and grades.",
                      "input_schema": SCHEMA}],
              messages=[{"role": "user", "content": text}])
    # newer models reject temperature and/or forced tool choice: fall back step by step (auto = model decides)
    global _claude_variant
    variants = [{"tool_choice": {"type": "tool", "name": "record_labels"}, "temperature": 0},
                {"tool_choice": {"type": "tool", "name": "record_labels"}},
                {"tool_choice": {"type": "auto"}}]
    last = None
    for i in range(_claude_variant, len(variants)):
        try:
            msg = _claude.messages.create(**kw, **variants[i])
            _claude_variant = i          # remember what this model accepts; later calls start there
            break
        except (anthropic.BadRequestError, TypeError) as e:
            last = e
            if isinstance(e, anthropic.BadRequestError) and not any(w in str(e) for w in ("temperature", "tool_choice")):
                raise                    # a different problem (e.g. bad schema): don't mask it
    else:
        raise last
    usage = {"in": msg.usage.input_tokens, "out": msg.usage.output_tokens}
    for b in msg.content:
        if getattr(b, "type", "") == "tool_use":
            return b.input, usage
    # no tool call: accept a JSON object written as plain text
    txt = "".join(getattr(b, "text", "") for b in msg.content)
    m = re.search(r"\{.*\}", txt, re.S)
    if m:
        return json.loads(m.group(0)), usage
    raise ValueError("no tool call or JSON in Claude response")

def call_gemini(text):
    global _gemini
    from google import genai
    from google.genai import types
    _gemini = _gemini or genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    prompt_schema = types.GenerateContentConfig(
        system_instruction=SYSTEM + "\n\nReturn JSON that follows this schema exactly:\n" + json.dumps(SCHEMA),
        temperature=0, max_output_tokens=16000, response_mime_type="application/json")
    try:
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0, max_output_tokens=16000,
                                          response_mime_type="application/json", response_json_schema=SCHEMA)
    except Exception:   # older SDK without response_json_schema: schema goes in the prompt instead
        cfg = prompt_schema
    try:
        r = _gemini.models.generate_content(model=GEMINI_MODEL, contents=text, config=cfg)
    except Exception as e:
        if cfg is prompt_schema or "schema" not in str(e).lower():
            raise
        r = _gemini.models.generate_content(model=GEMINI_MODEL, contents=text, config=prompt_schema)
    txt = (r.text or "").strip()
    txt = re.sub(r"^```(json)?|```$", "", txt).strip()
    u = getattr(r, "usage_metadata", None)
    return json.loads(txt), {"in": getattr(u, "prompt_token_count", None), "out": getattr(u, "candidates_token_count", None)}

def _strict(schema):
    # OpenAI strict structured outputs: every object needs additionalProperties=false and all keys required
    if isinstance(schema, dict):
        out = {k: _strict(v) for k, v in schema.items()}
        if out.get("type") == "object":
            out["additionalProperties"] = False
            out["required"] = list(out.get("properties", {}))
        return out
    if isinstance(schema, list):
        return [_strict(x) for x in schema]
    return schema

_openai = None
def call_openai(text):
    global _openai
    import openai
    _openai = _openai or openai.OpenAI()
    fmt = {"type": "json_schema", "json_schema": {"name": "knee_labels", "schema": _strict(SCHEMA), "strict": True}}
    kw = dict(model=OPENAI_MODEL, response_format=fmt,
              messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}])
    try:
        r = _openai.chat.completions.create(temperature=0, **kw)
    except openai.BadRequestError as e:
        if "temperature" not in str(e).lower():
            raise
        r = _openai.chat.completions.create(**kw)       # reasoning models only accept the default temperature
    msg = r.choices[0].message
    if getattr(msg, "refusal", None):
        raise ValueError(f"refusal: {msg.refusal[:200]}")
    u = r.usage
    return json.loads(msg.content), {"in": u.prompt_tokens, "out": u.completion_tokens}

CALLERS = {"claude": call_claude, "openai": call_openai, "gemini": call_gemini}

In [ ]:
# ---- resumable parallel runner ------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed

def jsonl_path(model):
    # one file per provider AND model id, so switching e.g. OPENAI_MODEL starts a fresh file instead of resuming the old one
    mid = {"claude": CLAUDE_MODEL, "openai": OPENAI_MODEL, "gemini": GEMINI_MODEL}[model]
    return OUT / f"raw_{model}_{re.sub(r'[^A-Za-z0-9._-]', '_', mid)}.jsonl"

def done_ids(model):
    p = jsonl_path(model)
    if not p.exists():
        return set()
    ok = set()
    for line in open(p):
        try:
            r = json.loads(line)
        except Exception:
            continue
        if r.get("ok"):
            ok.add(r["uid"])
    return ok

def run(model, ids, tries=6):
    fn, lock = CALLERS[model], threading.Lock()
    todo = [u for u in ids if u not in done_ids(model)]
    print(f"{model}: {len(ids) - len(todo):,} already done, {len(todo):,} to go", flush=True)
    if not todo:
        return
    stats = {"ok": 0, "bad": 0, "in": 0, "out": 0}
    t0 = time.time()

    def one(uid):
        err = None
        for k in range(tries):
            try:
                data, usage = fn(report_text(uid))
                row = validate(data)
                return {"uid": uid, "ok": True, "row": row, "usage": usage}
            except Exception as e:
                err = f"{type(e).__name__}: {str(e)[:200]}"
                time.sleep(min(60, 2 ** k + random.random()))
        return {"uid": uid, "ok": False, "error": err}

    with open(jsonl_path(model), "a") as f, ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(one, u) for u in todo]
        for i, fu in enumerate(as_completed(futs), 1):
            rec = fu.result()
            with lock:
                f.write(json.dumps(rec) + "\n"); f.flush()
                stats["ok" if rec["ok"] else "bad"] += 1
                if rec["ok"] and rec.get("usage"):
                    stats["in"] += rec["usage"].get("in") or 0
                    stats["out"] += rec["usage"].get("out") or 0
                if not rec["ok"] and stats["bad"] <= 3:
                    print("  failed:", rec["uid"][-12:], rec["error"], flush=True)
            if i % 50 == 0 or i == len(todo):
                el = time.time() - t0
                print(f"  {model}: {i:,}/{len(todo):,} | ok {stats['ok']} bad {stats['bad']} | "
                      f"tokens in {stats['in']:,} out {stats['out']:,} | {el/60:.1f} min, eta {(len(todo)-i)*el/i/60:.0f} min", flush=True)

def load(model):
    rows = {}
    p = jsonl_path(model)
    if p.exists():
        for line in open(p):
            r = json.loads(line)
            if r.get("ok"):
                rows[r["uid"]] = r["row"]
    return pd.DataFrame.from_dict(rows, orient="index")

for m in MODELS:
    run(m, IDS)

In [ ]:
# ---- evaluate: gold (blind) and silver-2 hand labels ------------------------------------------
from sklearn.metrics import roc_auc_score
P = {m: load(m) for m in MODELS}
for m, d in P.items():
    print(f"{m}: {len(d):,} reports parsed | not single knee: {int((~d['single_knee'].astype(bool)).sum())}")

def probs(d):
    return d[[f"p_{k}" for k in LABELS]].set_axis(COMP, axis=1)

srcs = {m: probs(d) for m, d in P.items()}
if len(srcs) == 2:
    a, b = srcs.values()
    ids = a.index.intersection(b.index)
    srcs["average"] = (a.loc[ids] + b.loc[ids]) / 2

def auc_table(truth, extra=None):
    out = {}
    cands = dict(srcs)
    if extra is not None:
        cands = {**extra, **cands}
    for name, s in cands.items():
        ids = truth.index.intersection(s.index)
        y = truth.loc[ids] > 0.5
        out[name] = {t: roc_auc_score(y[t], s.loc[ids, t]) if 0 < y[t].sum() < len(y) else np.nan for t in COMP}
    df = pd.DataFrame(out)
    df.loc["MACRO"] = df.mean()
    df.loc["weak five"] = df.loc[["MCL", "Lateral Meniscus", "Lateral OA", "PF OA", "Synovitis"]].mean()
    df.loc["n positives"] = [int(((truth.loc[truth.index.intersection(s.index)] > .5).sum()).sum()) for s in cands.values()]
    return df.round(3)

gold_truth = v31.loc[GOLD, COMP]
print("\n=== vs GOLD (58, image-derived radiologist labels; blind) ===")
print(auc_table(gold_truth).to_string())
s2_truth = s2.loc[s2.index.intersection(v31.index)]
print("\n=== vs SILVER-2 hand labels (159; current extraction v3.1 shown for comparison) ===")
print(auc_table(s2_truth, extra={"v3.1 (current)": v31.loc[s2_truth.index, COMP]}).to_string())

if len(MODELS) == 2:
    a, b = P[MODELS[0]], P[MODELS[1]]
    ids = a.index.intersection(b.index)
    gap = pd.DataFrame({t: (a.loc[ids, f"p_{k}"] - b.loc[ids, f"p_{k}"]).abs() for k, t in zip(LABELS, COMP)})
    print(f"\n{MODELS[0]} vs {MODELS[1]}: mean |p difference| and share of cells differing by >= 0.4")
    print(pd.DataFrame({"mean_abs_diff": gap.mean(), "share_ge_0.4": (gap >= 0.4).mean()}).round(3).to_string())

In [ ]:
# ---- export: per-model files, combined labels, disagreement list --------------------------------
def export():
    files = {}
    for m, d in P.items():
        f = OUT / f"llm_{m}_{STAGE}.csv"
        d.rename_axis("StudyInstanceUID").to_csv(f)
        files[m] = f
    if len(P) == 2:
        (na, a), (nb, b) = list(P.items())
        ids = a.index.union(b.index)
        comb = pd.DataFrame(index=ids)
        for k, t in zip(LABELS, COMP):
            pa = a[f"p_{k}"].reindex(ids)
            pb = b[f"p_{k}"].reindex(ids)
            comb[t] = pd.concat([pa, pb], axis=1).mean(axis=1)      # average where both answered
        comb.rename_axis("StudyInstanceUID").to_csv(OUT / f"llm_combined_{STAGE}.csv")
        rows = []
        both = a.index.intersection(b.index)
        for k, t in zip(LABELS, COMP):
            dif = (a.loc[both, f"p_{k}"] - b.loc[both, f"p_{k}"]).abs()
            for uid in dif[dif >= 0.4].index:
                rows.append({"StudyInstanceUID": uid, "label": t, f"p_{na}": a.at[uid, f"p_{k}"], f"p_{nb}": b.at[uid, f"p_{k}"],
                             f"ev_{na}": a.at[uid, f"ev_{k}"], f"ev_{nb}": b.at[uid, f"ev_{k}"],
                             "report_english": str(v31.at[uid, "report_english"])[:1500]})
        pd.DataFrame(rows).to_csv(OUT / f"llm_disagreements_{STAGE}.csv", index=False)
        print(f"combined: {len(comb):,} studies | disagreements (|diff| >= 0.4): {len(rows):,} cells in "
              f"{len({r['StudyInstanceUID'] for r in rows}):,} studies")
    print("written to", OUT, ":", sorted(p.name for p in OUT.glob("llm_*.csv")))

export()

## Adjudication: a third model settles the uncertain cells
Sends only the cells where the two readers disagree (|pA - pB| >= `ADJ_GAP`), plus weak-label cells where both sit near 0.5.
The adjudicator sees the report, the rubric and both readers' probabilities and quotes (as "reader A/B", unnamed), and returns one probability per listed cell.
Run the extraction above for both models first. Resumable like the extraction. Run on the pilot first: the last cell compares average vs final against gold.

In [ ]:
# ---- adjudication settings + cell selection ---------------------------------------------------
ADJ_PROVIDER = "claude"               # "claude" or "openai"
ADJ_MODEL = "claude-opus-5-5"         # stronger than both readers
ADJ_GAP = 0.25                        # send a cell when the readers differ by at least this much
WEAK = ["mcl", "lateral_meniscus", "lateral_oa", "pf_oa", "synovitis"]
NEAR = (0.35, 0.65)                   # ...or when both readers sit in this band on a weak label
ADJ_WORKERS = 6

assert len(MODELS) == 2, "adjudication needs both readers"
A, B = P[MODELS[0]], P[MODELS[1]]
both = A.index.intersection(B.index)
todo_cells = {}
for k in LABELS:
    pa, pb = A.loc[both, f"p_{k}"], B.loc[both, f"p_{k}"]
    sel = (pa - pb).abs() >= ADJ_GAP
    if k in WEAK:
        sel |= pa.between(*NEAR) & pb.between(*NEAR)
    for uid in sel[sel].index:
        todo_cells.setdefault(uid, []).append(k)
n_cells = sum(len(v) for v in todo_cells.values())
print(f"{n_cells:,} cells in {len(todo_cells):,} studies to adjudicate "
      f"({n_cells / (len(both) * len(LABELS)):.1%} of all cells)")
print(pd.Series([k for v in todo_cells.values() for k in v]).value_counts().to_string())

In [ ]:
# ---- adjudicator prompt, schema, caller, resumable runner ----------------------------------------
ADJ_SYSTEM = (SYSTEM + "\n\n## Adjudication task\n"
    "Two independent readers already applied the rubric above to this report and disagreed, or were both unsure, "
    "on the findings listed below. For each listed finding, re-read the report yourself, check both readers' "
    "probabilities and quotes against the criteria and calibration anchors, and give the final probability that the "
    "image readers marked it POSITIVE. You may agree with either reader or choose a different value. "
    "Watch for the usual errors: medial/lateral or internal/external swaps, negations attached to the wrong structure, "
    "acute vs chronic, fluid volume vs synovitis, contusion vs fracture, and anchors applied to the wrong wording. "
    "Decide every listed finding exactly once; do not add findings that are not listed. "
    "reason: one short sentence. Answer only through the required JSON structure.")

def adj_schema(keys):
    return {"type": "object",
            "properties": {"decisions": {"type": "array", "items": {
                "type": "object",
                "properties": {"label": {"type": "string", "enum": list(keys)},
                               "p": {"type": "number"},
                               "reason": {"type": "string"}},
                "required": ["label", "p", "reason"]}}},
            "required": ["decisions"]}

def adj_prompt(uid, keys):
    lines = [report_text(uid), "", "## Findings to adjudicate"]
    for k in keys:
        lines.append(f"- {k}: reader A p={A.at[uid, f'p_{k}']:.2f} ({A.at[uid, f'st_{k}']}), quote: \"{A.at[uid, f'ev_{k}']}\" | "
                     f"reader B p={B.at[uid, f'p_{k}']:.2f} ({B.at[uid, f'st_{k}']}), quote: \"{B.at[uid, f'ev_{k}']}\"")
    return "\n".join(lines)

_adj_variant = 0
def call_adjudicator(text, schema):
    global _claude, _openai, _adj_variant
    if ADJ_PROVIDER == "openai":
        import openai
        _openai = _openai or openai.OpenAI()
        fmt = {"type": "json_schema", "json_schema": {"name": "adjudication", "schema": _strict(schema), "strict": True}}
        r = _openai.chat.completions.create(model=ADJ_MODEL, response_format=fmt,
                messages=[{"role": "system", "content": ADJ_SYSTEM}, {"role": "user", "content": text}])
        u = r.usage
        return json.loads(r.choices[0].message.content), {"in": u.prompt_tokens, "out": u.completion_tokens}
    import anthropic
    _claude = _claude or anthropic.Anthropic()
    kw = dict(model=ADJ_MODEL, max_tokens=3000,
              system=[{"type": "text", "text": ADJ_SYSTEM + "\n\nAlways answer by calling the record_decisions tool exactly once.",
                       "cache_control": {"type": "ephemeral"}}],
              tools=[{"name": "record_decisions", "description": "Record one final probability per listed finding.",
                      "input_schema": schema}],
              messages=[{"role": "user", "content": text}])
    variants = [{"tool_choice": {"type": "tool", "name": "record_decisions"}}, {"tool_choice": {"type": "auto"}}]
    last = None
    for i in range(_adj_variant, len(variants)):
        try:
            msg = _claude.messages.create(**kw, **variants[i]); _adj_variant = i; break
        except (anthropic.BadRequestError, TypeError) as e:
            last = e
            if isinstance(e, anthropic.BadRequestError) and "tool_choice" not in str(e):
                raise
    else:
        raise last
    usage = {"in": msg.usage.input_tokens, "out": msg.usage.output_tokens}
    for b in msg.content:
        if getattr(b, "type", "") == "tool_use":
            return b.input, usage
    m = re.search(r"\{.*\}", "".join(getattr(b, "text", "") for b in msg.content), re.S)
    if m:
        return json.loads(m.group(0)), usage
    raise ValueError("no tool call or JSON in adjudicator response")

def adj_path():
    return OUT / f"raw_adjudicator_{re.sub(r'[^A-Za-z0-9._-]', '_', ADJ_MODEL)}.jsonl"

def adj_load():
    # uid -> {label: (p, reason)}; later records win, so a re-run with new cells adds to earlier ones
    out = {}
    if adj_path().exists():
        for line in open(adj_path()):
            try:
                r = json.loads(line)
            except Exception:
                continue
            if r.get("ok"):
                out.setdefault(r["uid"], {}).update({k: tuple(v) for k, v in r["dec"].items()})
    return out

def adj_one(uid, keys, tries=6):
    err = None
    for t in range(tries):
        try:
            d, usage = call_adjudicator(adj_prompt(uid, keys), adj_schema(keys))
            if isinstance(d, str):
                d = json.loads(d)
            dec = {}
            for x in d["decisions"]:
                p = float(x["p"])
                if x["label"] in keys and np.isfinite(p):
                    dec[x["label"]] = (min(1.0, max(0.0, p)), str(x.get("reason", ""))[:300])
            miss = set(keys) - set(dec)
            if miss:
                raise ValueError(f"undecided: {sorted(miss)}")
            return {"uid": uid, "ok": True, "dec": dec, "usage": usage}
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
            time.sleep(min(60, 2 ** t + random.random()))
    return {"uid": uid, "ok": False, "error": err}

def run_adjudication():
    have = adj_load()
    jobs = {u: [k for k in ks if k not in have.get(u, {})] for u, ks in todo_cells.items()}
    jobs = {u: ks for u, ks in jobs.items() if ks}
    print(f"adjudicator {ADJ_MODEL}: {len(todo_cells) - len(jobs):,} studies done, {len(jobs):,} to go", flush=True)
    st, lock, t0 = {"ok": 0, "bad": 0, "in": 0, "out": 0}, threading.Lock(), time.time()
    with open(adj_path(), "a") as f, ThreadPoolExecutor(ADJ_WORKERS) as ex:
        futs = [ex.submit(adj_one, u, ks) for u, ks in jobs.items()]
        for i, fu in enumerate(as_completed(futs), 1):
            rec = fu.result()
            with lock:
                f.write(json.dumps(rec) + "\n"); f.flush()
                st["ok" if rec["ok"] else "bad"] += 1
                if rec["ok"]:
                    st["in"] += rec["usage"].get("in") or 0; st["out"] += rec["usage"].get("out") or 0
                elif st["bad"] <= 3:
                    print("  failed:", rec["uid"][-12:], rec["error"], flush=True)
            if i % 50 == 0 or i == len(jobs):
                el = time.time() - t0
                print(f"  {i:,}/{len(jobs):,} | ok {st['ok']} bad {st['bad']} | tokens in {st['in']:,} out {st['out']:,} | "
                      f"{el/60:.1f} min, eta {(len(jobs)-i)*el/i/60:.0f} min", flush=True)

run_adjudication()

In [ ]:
# ---- final labels: average of the two readers, adjudicated cells replaced ----------------------------
ADJ = adj_load()
ids = A.index.union(B.index)
avg = pd.DataFrame({t: pd.concat([A[f"p_{k}"].reindex(ids), B[f"p_{k}"].reindex(ids)], axis=1).mean(axis=1)
                    for k, t in zip(LABELS, COMP)})
final = avg.copy()
rows = []
for uid, dec in ADJ.items():
    for k, (p, why) in dec.items():
        if uid in todo_cells and k in todo_cells[uid]:          # only cells selected under the current settings
            t = COMP[LABELS.index(k)]
            rows.append({"StudyInstanceUID": uid, "label": t, f"p_{MODELS[0]}": A.at[uid, f"p_{k}"],
                         f"p_{MODELS[1]}": B.at[uid, f"p_{k}"], "p_average": avg.at[uid, t], "p_final": p, "reason": why})
            final.at[uid, t] = p
adj_df = pd.DataFrame(rows)
final.rename_axis("StudyInstanceUID").to_csv(OUT / f"llm_final_{STAGE}.csv")
adj_df.to_csv(OUT / f"llm_adjudicated_{STAGE}.csv", index=False)
print(f"final: {len(final):,} studies | adjudicated cells applied: {len(adj_df):,} of {n_cells:,} selected")
if len(adj_df):
    print("mean |final - average| on adjudicated cells:", round((adj_df.p_final - adj_df.p_average).abs().mean(), 3))

if STAGE == "pilot":
    cmp = {"average": avg, "final (adjudicated)": final}
    for name, truth in [("GOLD (58, blind)", gold_truth), ("SILVER-2 (159)", s2_truth)]:
        out = {}
        for n, s in cmp.items():
            ii = truth.index.intersection(s.index)
            y = truth.loc[ii] > 0.5
            out[n] = {t: roc_auc_score(y[t], s.loc[ii, t]) if 0 < y[t].sum() < len(y) else np.nan for t in COMP}
        df = pd.DataFrame(out)
        df.loc["MACRO"] = df.mean()
        df.loc["weak five"] = df.loc[["MCL", "Lateral Meniscus", "Lateral OA", "PF OA", "Synovitis"]].mean()
        df["delta"] = df.iloc[:, 1] - df.iloc[:, 0]
        print(f"\n=== {name}: average vs adjudicated ===")
        print(df.round(3).to_string())
        if len(adj_df):   # on adjudicated cells only: which side moved toward the truth?
            m = adj_df[adj_df.StudyInstanceUID.isin(truth.index)].copy()
            if len(m):
                m["y"] = [float(truth.at[u, t] > .5) for u, t in zip(m.StudyInstanceUID, m.label)]
                print(f"  adjudicated cells with truth: {len(m)} | mean abs error average "
                      f"{(m.p_average - m.y).abs().mean():.3f} -> final {(m.p_final - m.y).abs().mean():.3f}")